# Focus Fatigue Analysis
## Cognitive Fatigue Detection in Football Defence

This notebook analyses the unified fatigue dataset, comparing defensive-quality signals across different pressure exposure levels and tracking temporal fatigue trends.

**Data sources:** Tracking data (Stats Perform, 25fps) processed through Model 1 (Pressure Exposure) and 4 defensive-quality signals.

## 1. Load & Explore the Unified Dataset

Loading the real unified parquet file and examining its structure, basic statistics, and coverage.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
import os, warnings
warnings.filterwarnings("ignore")

# Set up paths
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "."))
DATA_PATH = os.path.join(REPO_ROOT, "outputs", "unified_fatigue_dataset.parquet")
OUTPUT_DIR = os.path.join(REPO_ROOT, "outputs", "analysis")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plot aesthetics
plt.rcParams.update({
    "figure.figsize": (12, 6), "font.size": 11, "axes.titlesize": 14,
    "axes.labelsize": 12, "figure.dpi": 150, "savefig.dpi": 150,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.3,
})

print(f"Data file exists: {os.path.exists(DATA_PATH)}")

In [1]:
# Load data
df = pd.read_parquet(DATA_PATH)

signal_cols = ["positional_drift", "shift_latency", "pressing_accuracy", "transition_latency"]

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\n--- Basic Stats ---")
print(f"Number of matches: {df['game_id'].nunique()}  ({sorted(df['game_id'].unique())})")
print(f"Number of players: {df['player_id'].nunique()}")
print(f"Number of blocks: {df['block_id'].nunique()}")
print(f"Number of phases: {sorted(df['phase'].unique())}")
print(f"Players per match: {df.groupby('game_id')['player_id'].nunique().to_dict()}")

print(f"\n--- Signal Ranges ---")
for col in signal_cols:
    print(f"  {col}: min={df[col].min():.3f}, max={df[col].max():.3f}, "
          f"mean={df[col].mean():.3f}, std={df[col].std():.3f}")

print(f"\n--- Pressure Category Distribution ---")
print(df["pressure_category"].value_counts().to_string())

print(f"\n--- NaN Check ---")
print(f"Total NaN values: {df.isna().sum().sum()}")

print(f"\n--- Dtypes ---")
print(df.dtypes.to_string())

## 2. Per-Player Baseline Deviation

We define each player's **baseline** as the first 3 blocks of the match (opening 15 minutes, block indices _0, _1, _2). For every subsequent block, we compute a **z-score** (deviation from the player's own baseline mean, in units of baseline standard deviation). This helps track fatigue-related signal degradation relative to each player's fresh state.

In [1]:
# Parse numeric block index from block_id (e.g. '1_0' -> 0)
df["block_num"] = df["block_id"].apply(lambda b: int(b.split("_")[1]))

# Baseline: first 3 blocks (indices 0, 1, 2)
baseline_mask = df["block_num"].isin([0, 1, 2])

# Per-player baseline stats
baseline_stats = df[baseline_mask].groupby("player_id")[signal_cols].agg(["mean", "std"])
print("Baseline stats (mean over first 3 blocks per player):")
print(baseline_stats.describe().round(3))

# Compute z-scores for every row
# Compute baseline stats per player, then merge back to full dataframe
baseline_df = df[baseline_mask].groupby("player_id")[signal_cols].agg(["mean", "std"])
baseline_df.columns = [f"{c}_{stat}" for c in signal_cols for stat in ["mean", "std"]]
baseline_df = baseline_df.reset_index()

# Merge baseline stats into full dataframe
df = df.merge(baseline_df, on="player_id", how="left")

# Compute z-scores
for col in signal_cols:
    z_col = f"{col}_z"
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"
    # Avoid division by zero: use NaN where std is effectively 0
    safe_std = df[std_col].replace(0, np.nan)
    df[z_col] = (df[col] - df[mean_col]) / safe_std

print(f"\nZ-score ranges (non-baseline blocks only):")
for col in signal_cols:
    z_col = f"{col}_z"
    vals = df.loc[~baseline_mask, z_col].dropna()
    if len(vals) > 0:
        print(f"  {col}: z-range [{vals.min():.2f}, {vals.max():.2f}], mean={vals.mean():.3f}")

print(f"\nNaN z-scores (zero-std baseline): {df[signal_cols[0]+'_z'].isna().sum()}")

## 3. Pressure Category Comparison

We compare the four defensive-quality signals across **High**, **Medium**, and **Low** pressure blocks. Boxplots illustrate distribution differences, and a summary table shows mean +/- std for each signal x pressure category.

In [1]:
# Pressure category boxplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

signal_labels = {
    "positional_drift": "Positional Drift (m)",
    "shift_latency": "Shift Latency (s)",
    "pressing_accuracy": "Pressing Accuracy (0-1)",
    "transition_latency": "Transition Latency (s)"
}

cat_order = ["low", "medium", "high"]
cat_labels = ["Low", "Medium", "High"]
cat_colors = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}

for i, col in enumerate(signal_cols):
    ax = axes[i]
    data = [df[df["pressure_category"] == c][col].values for c in cat_order]
    bp = ax.boxplot(data, tick_labels=cat_labels,
                    patch_artist=True, widths=0.5)
    for patch, c in zip(bp["boxes"], cat_order):
        patch.set_facecolor(cat_colors[c])
        patch.set_alpha(0.6)
    ax.set_title(signal_labels[col])
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Figure 1: Signal Distributions by Pressure Category", fontsize=16, y=1.02)
plt.tight_layout()
fig1 = os.path.join(OUTPUT_DIR, "fig1_boxplot_pressure_signals.png")
plt.savefig(fig1)
plt.close()
print(f"Saved: {fig1}")

print("\n--- Signal Means by Pressure Category ---")
print(df.groupby("pressure_category")[signal_cols].agg(["mean", "std"]).round(3).to_string())

print("\n--- Signal Median by Pressure Category ---")
print(df.groupby("pressure_category")[signal_cols].median().round(3).to_string())

## 4. Statistical Tests: High vs Low Pressure

We run **paired statistical tests** comparing each signal between high-pressure and low-pressure blocks. Since each player appears in both conditions, we pair by player:
- For each player, compute their mean signal value during high-pressure blocks
- And their mean during low-pressure blocks
- Apply the **Wilcoxon signed-rank test** (non-parametric, appropriate for paired data)
- We also report **Cohen's d** for effect size

In [1]:
def cohens_d(a, b):
    diff = np.array(a) - np.array(b)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-10)

print("=" * 70)
print("PAIRED WILCOXON TEST: HIGH vs LOW PRESSURE (paired by player)")
print("=" * 70)

results = []
for col in signal_cols:
    high = df[df["pressure_category"]=="high"].groupby("player_id")[col].mean()
    low  = df[df["pressure_category"]=="low"].groupby("player_id")[col].mean()
    common = high.index.intersection(low.index)
    if len(common) < 3:
        print(f"  {col}: Too few common players (n={len(common)}), skipping")
        continue
    h, l = high[common].values, low[common].values
    stat, p = stats.wilcoxon(h, l, alternative="two-sided")
    d = cohens_d(h, l)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    direction = "HIGH > LOW" if np.mean(h-l) > 0 else "HIGH < LOW"
    results.append({"signal": col, "n": len(common),
        "high_mean": np.mean(h), "low_mean": np.mean(l),
        "diff": np.mean(h-l), "cohens_d": d,
        "W": stat, "p": p, "sig": sig, "dir": direction})
    print(f"\n  {col} (n={len(common)}): High={np.mean(h):.4f} vs Low={np.mean(l):.4f}")
    print(f"    Diff={np.mean(h-l):.4f}, W={stat:.1f}, p={p:.4f} {sig}")
    print(f"    Cohen's d={d:.3f} ({direction})")

print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
if results:
    print(pd.DataFrame(results).round(4).to_string(index=False))

In [1]:
# Additional comparisons: High vs Medium, Medium vs Low
def paired_test(df, col, a_cat, b_cat, a_label, b_label):
    a = df[df["pressure_category"]==a_cat].groupby("player_id")[col].mean()
    b = df[df["pressure_category"]==b_cat].groupby("player_id")[col].mean()
    common = a.index.intersection(b.index)
    if len(common) < 3: return None
    av, bv = a[common].values, b[common].values
    try:
        w, p = stats.wilcoxon(av, bv, alternative="two-sided")
    except ValueError:
        return None
    d = cohens_d(av, bv)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
    return {"comparison": f"{a_label} vs {b_label}", "signal": col,
            "n": len(common), "diff": np.mean(av-bv), "cohens_d": d,
            "p": p, "sig": sig}

print("\n" + "=" * 70)
print("ADDITIONAL PAIRED COMPARISONS")
print("=" * 70)
extra = []
for col in signal_cols:
    for (a_c, b_c, a_l, b_l) in [("high","low","High","Low"),
                                   ("high","medium","High","Medium"),
                                   ("medium","low","Medium","Low")]:
        r = paired_test(df, col, a_c, b_c, a_l, b_l)
        if r: extra.append(r)
if extra:
    print(pd.DataFrame(extra).round(4).to_string(index=False))

## 5. Temporal Trends Across the Match

We track how each signal evolves over 18 time blocks (90 minutes + stoppage time). Per-player lines (grey) show individual trajectories, the bold line shows the across-player average. Red-shaded areas indicate blocks where >50% of players were under high pressure. The dashed vertical line marks half time.

In [1]:
# Chronological block ordering
block_order = sorted(df["block_id"].unique(),
                     key=lambda b: (int(b.split("_")[0]), int(b.split("_")[1])))
# Short labels (just the numeric index)
block_short = [b.split("_")[1] for b in block_order]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(signal_cols):
    ax = axes[i]
    # Per-player lines
    for pid in df["player_id"].unique():
        pd_sub = df[df["player_id"]==pid].set_index("block_id").reindex(block_order)
        ax.plot(range(18), pd_sub[col].values,
                color="gray", alpha=0.12, linewidth=0.7)
    # Average
    avg = df.groupby("block_id")[col].mean().reindex(block_order)
    ax.plot(range(18), avg.values, color="#e74c3c", linewidth=2.5, label="Player Average")
    # Highlight high-pressure blocks
    for bi, bid in enumerate(block_order):
        pct_high = (df[df["block_id"]==bid]["pressure_category"]=="high").mean()
        if pct_high > 0.5:
            ax.axvspan(bi-0.5, bi+0.5, color="red", alpha=0.08)
    # Half-time line
    ax.axvline(x=8.5, color="black", linestyle="--", alpha=0.5)
    ax.set_xticks(range(18))
    ax.set_xticklabels(block_short, rotation=45, fontsize=8)
    ax.set_xlabel("Time Block (5 min each)")
    ax.set_ylabel(signal_labels[col])
    ax.set_title(f"{signal_labels[col]} — Across-Match Trend")
    ax.grid(True, alpha=0.3, axis="y")
    leg = [plt.Line2D([0],[0],color="#e74c3c",lw=2.5,label="Player Average"),
           Patch(facecolor="red", alpha=0.08, label=">50% High Pressure"),
           plt.Line2D([0],[0],color="black",ls="--",alpha=0.5,label="Half Time")]
    ax.legend(handles=leg, fontsize=9, loc="upper left")

plt.suptitle("Figure 2: Temporal Progression of Defensive Signals Across the Match", fontsize=15, y=1.02)
plt.tight_layout()
fig2 = os.path.join(OUTPUT_DIR, "fig2_temporal_trends_signals.png")
plt.savefig(fig2)
plt.close()
print(f"Saved: {fig2}")

In [1]:
# Z-score temporal trends (deviation from first-15-min baseline)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(signal_cols):
    ax = axes[i]
    zcol = f"{col}_z"
    for pid in df["player_id"].unique():
        pd_sub = df[df["player_id"]==pid].set_index("block_id").reindex(block_order)
        ax.plot(range(18), pd_sub[zcol].values,
                color="gray", alpha=0.12, linewidth=0.7)
    avg_z = df.groupby("block_id")[zcol].mean().reindex(block_order)
    ax.plot(range(18), avg_z.values, color="#2980b9", linewidth=2.5, label="Average Z-Score")
    ax.axhline(y=0, color="green", ls=":", alpha=0.6, label="Baseline")
    ax.axvline(x=8.5, color="black", ls="--", alpha=0.5)
    ax.set_xticks(range(18))
    ax.set_xticklabels(block_short, rotation=45, fontsize=8)
    ax.set_xlabel("Time Block (5 min each)")
    ax.set_ylabel(f"{signal_labels[col]} Z-Score")
    ax.set_title(f"{signal_labels[col]} — Deviation from First-15-Min Baseline")
    ax.grid(True, alpha=0.3, axis="y")
    ax.legend(fontsize=9, loc="upper left")

plt.suptitle("Figure 3: Signal Z-Score Progression (Deviation from Player Baseline)", fontsize=15, y=1.02)
plt.tight_layout()
fig3 = os.path.join(OUTPUT_DIR, "fig3_zscore_temporal_trends.png")
plt.savefig(fig3)
plt.close()
print(f"Saved: {fig3}")

In [1]:
# Pressure category distribution across blocks
fig, ax = plt.subplots(figsize=(14, 5))
pct_by_block = (df.groupby(["block_id","pressure_category"]).size().unstack(fill_value=0)
                .reindex(block_order))
pct_by_block = pct_by_block.div(pct_by_block.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(block_order))
for c in cat_order:
    vals = pct_by_block[c].values if c in pct_by_block.columns else np.zeros(len(block_order))
    ax.bar(range(len(block_order)), vals, bottom=bottom, color=cat_colors[c],
           label=c.capitalize(), alpha=0.8, width=0.7)
    bottom += vals

ax.axvline(x=8.5, color="black", ls="--", alpha=0.5, lw=1.5)
ax.set_xticks(range(len(block_order)))
ax.set_xticklabels(block_short, rotation=45)
ax.set_xlabel("Time Block (5 min each)")
ax.set_ylabel("% of Player-Blocks")
ax.set_title("Figure 4: Pressure Category Distribution Across Match Timeline")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")
ax.text(4, 50, "First Half", ha="center", fontsize=10, style="italic", alpha=0.6)
ax.text(13, 50, "Second Half", ha="center", fontsize=10, style="italic", alpha=0.6)

plt.tight_layout()
fig4 = os.path.join(OUTPUT_DIR, "fig4_pressure_distribution_timeline.png")
plt.savefig(fig4)
plt.close()
print(f"Saved: {fig4}")

In [1]:
# Spearman correlation between signals under high vs low pressure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
short_names = [s.replace("_"," ").title() for s in signal_cols]

for idx, cat in enumerate(["low", "high"]):
    ax = axes[idx]
    corr = df[df["pressure_category"]==cat][signal_cols].corr(method="spearman")
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(short_names, fontsize=9)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center",
                    color="white" if abs(corr.values[i,j])>0.5 else "black", fontsize=9)
    ax.set_title(f"{cat.capitalize()} Pressure — Signal Correlations")

plt.suptitle("Figure 5: Spearman Correlation Between Signals (by Pressure Condition)", fontsize=14, y=1.02)
plt.tight_layout()
fig5 = os.path.join(OUTPUT_DIR, "fig5_correlation_high_vs_low_pressure.png")
plt.savefig(fig5)
plt.close()
print(f"Saved: {fig5}")

## 6. Key Findings Summary

This section prints a comprehensive summary of the analysis findings.

In [1]:
print("=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

n_matches = df['game_id'].nunique()
n_players = df['player_id'].nunique()
n_blocks = df['block_id'].nunique()
print(f"\nDataset: {n_matches} matches, {n_players} players, {n_blocks} time blocks")

print("\n--- Signal Ranges ---")
for col in signal_cols:
    print(f"  {col}: [{df[col].min():.3f}, {df[col].max():.3f}]")

print("\n--- Pressure Categories ---")
for cat in ["high", "medium", "low"]:
    cnt = len(df[df["pressure_category"]==cat])
    print(f"  {cat.capitalize():>7}: {cnt:>4} blocks ({cnt/len(df)*100:5.1f}%)")

print("\n--- Statistical Test Summary (High vs Low Pressure) ---")
if results:
    for r in results:
        sig_str = f"p={r['p']:.4f} {r['sig']}" if r['p']<0.05 else f"p={r['p']:.4f} (ns)"
        print(f"  {r['signal']}: Cohen's d={r['cohens_d']:.3f}, {sig_str}")

print("\n--- Temporal Trend Summary (first vs last 15 min) ---")
for col in signal_cols:
    early = df[df["block_num"].isin([0,1,2])][col].mean()
    late  = df[df["block_num"].isin([15,16,17])][col].mean()
    pct = ((late-early)/early)*100 if early != 0 else 0
    print(f"  {col}: Early={early:.3f}, Late={late:.3f}, change={pct:+.1f}% {'\u2191' if pct>0 else '\u2193'}")

print("\n--- Key Insights ---")
sig_results = [r for r in results if r["p"] < 0.05] if results else []
if sig_results:
    print("  Significant pressure effects found:")
    for r in sig_results:
        dirmap = "higher" if r["diff"]>0 else "lower"
        print(f"    - {r['signal']}: values are {dirmap} under high pressure"
              f" (d={r['cohens_d']:.2f}, p={r['p']:.4f})")
else:
    print("  No significant differences between high and low pressure blocks")

# Temporal fatigue
print("\n  Temporal fatigue effects (second-half deviation from baseline):")
for col in signal_cols:
    zcol = f"{col}_z"
    z = df[df["block_num"]>8][zcol].mean()
    if abs(z) > 0.3:
        print(f"    - {col}: second-half z={z:.2f} ({'above' if z>0 else 'below'} baseline)")

print("\n  Figures saved:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"    - outputs/analysis/{f}")
print("=" * 70)